# Parallel 06 — TP 手写 linear：把一层网络横着切开

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/parallel/06-tensor-parallel.ipynb)

- **日期**：2026-10-02
- **定位**：并行基础线第三篇。面试形式很明确：给一个两层 MLP，让你手写 Megatron 式的张量并行版，并验证与单卡数值对齐。
- **环境**：CPU + gloo，2 进程验证机制。

配套笔记：[06-tensor-parallel.md](06-tensor-parallel.md)


## 0. 因果链定位

DDP 切 batch、ZeRO 切存储，都没动**一层网络内部的计算**。当一层权重本身装不下（或一层的矩阵乘就是瓶颈），必须把 $ Y = XW $ 这个乘法拆开：

- **按列切** $ W = [W_1 | W_2] $ ：每 rank 算 $ XW_i $ ，得到半截输出，前向不通信；
- **按行切** $ W = [W_1; W_2] $ 、 $ X $ 同步按特征切：每 rank 算 $ X_iW_i $ ，是部分和，出口 all-reduce 相加。

Megatron 的关键组合：第一层列切 → GELU（逐元素，半截也能算）→ 第二层行切 → 只在出口通信一次。通信点选在"非线性之后、求和之处"。


## 1. 骨架：f 和 g 两个算子

Megatron 论文的记法：

- $ f $ ：前向恒等，反向 all-reduce（挂在 column-parallel 的**输入**端：前向大家拿同一份输入，反向每人只有半截梯度、要拼）
- $ g $ ：前向 all-reduce，反向恒等（挂在 row-parallel 的**输出**端）

用 `torch.autograd.Function` 把这两个算子写出来，TP 就剩组装了。


In [ ]:
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F

class f_op(torch.autograd.Function):
    """恒等算子 f：forward 原样返回 x，backward 把传回的 grad 跨 rank all-reduce 求和。

    forward(ctx, x): x 是任意形状的 float 张量；backward(ctx, grad): grad 同 x 形状。
    一个方向恒等、另一个方向规约，这个配对才在数学上等价（见正文推导）。"""
    @staticmethod
    def forward(ctx, x):
        return x
    @staticmethod
    def backward(ctx, grad):
        dist.all_reduce(grad)
        return grad

class g_op(torch.autograd.Function):
    """恒等算子 g：forward 把 x 跨 rank all-reduce 求和，backward 原样返回 grad。

    与 f_op 配对使用：f 放列切的入口、g 放行切的出口（见正文的组合推导）。"""
    @staticmethod
    def forward(ctx, x):
        dist.all_reduce(x)
        return x
    @staticmethod
    def backward(ctx, grad):
        return grad

class ColumnParallelLinear(torch.nn.Module):
    """列切 Linear：每 rank 只持有权重 W 的一部分输出列，各自那段输出互不依赖、不通信。

    __init__ 入参:
      d_in / d_out: 全量维度（只用于对账，实际形状以分片为准）；
      weight_shard: (d_out/N, d_in) —— 本 rank 的权重分片（F.linear 的 weight 约定）；
      bias_shard: (d_out/N,) —— 与该分片对应的偏置。
    forward: x (..., d_in) 全量输入（入口经 f_op）→ 本 rank 的 (..., d_out/N) 分片输出。"""
    def __init__(self, d_in, d_out, weight_shard, bias_shard):
        super().__init__()
        self.weight = torch.nn.Parameter(weight_shard.clone())
        self.bias = torch.nn.Parameter(bias_shard.clone())
    def forward(self, x):
        x = f_op.apply(x)
        return F.linear(x, self.weight, self.bias)

class RowParallelLinear(torch.nn.Module):
    """行切 Linear：每 rank 持有 W 的一部分输入行，吃分片输入、出全量（经 g 求和）。

    __init__ 入参:
      d_in / d_out: 全量维度；
      weight_shard: (d_out, d_in/N) —— 本 rank 的权重分片；
      bias_full: (d_out,) 全量偏置 —— 只在这一层、求和之后加一次（加错位置见调试清单）。
    forward: x (..., d_in/N) 分片输入 → (..., d_out) 全量输出。"""
    def __init__(self, d_in, d_out, weight_shard, bias_full):
        super().__init__()
        self.weight = torch.nn.Parameter(weight_shard.clone())
        self.bias = torch.nn.Parameter(bias_full.clone())
    def forward(self, x):
        return g_op.apply(F.linear(x, self.weight)) + self.bias


## 2. 组装两层 MLP，与单卡实现对拍

验证标准要狠：**前向输出对齐 + 输入梯度对齐 + 权重梯度（分片拼接后）对齐**。只对前向是抓不到 f/g 写反这类 bug 的。


In [ ]:
def run(rank, world_size):
    """mp.spawn 的入口，每个进程各跑一份本函数。

    rank: 本进程编号 [0, world_size)（真实训练里对应一张卡/一个节点角色）；
    world_size: 进程总数，本 notebook 固定 2（模拟 2 卡）。"""
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29504'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    torch.manual_seed(0)

    # 单卡参考实现（每个 rank 都建一份完整版，用同一组权重对拍）
    d_in, d_hid, d_out = 8, 16, 4
    l1 = torch.nn.Linear(d_in, d_hid)
    l2 = torch.nn.Linear(d_hid, d_out)
    x = torch.randn(4, d_in, requires_grad=True)
    x_ref = x.detach().clone().requires_grad_(True)

    y_ref = l2(F.gelu(l1(x_ref)))
    y_ref.sum().backward()

    # TP 版：W1 按列切（每 rank 8 个 hidden），W2 按行切（每 rank吃 8 个 hidden）
    w1_cols = l1.weight.data.chunk(world_size, dim=0)   # Linear 的 weight 是 (out, in)，列切 W 等价于行切 weight
    b1_cols = l1.bias.data.chunk(world_size, dim=0)
    w2_rows = l2.weight.data.chunk(world_size, dim=1)   # 行切 W 等价于列切 weight
    col = ColumnParallelLinear(d_in, d_hid, w1_cols[rank], b1_cols[rank])
    row = RowParallelLinear(d_hid, d_out, w2_rows[rank], l2.bias.data)

    y = row(F.gelu(col(x)))
    y.sum().backward()

    if rank == 0:
        print('前向输出最大差:', (y - y_ref).abs().max().item())
        print('输入梯度最大差:', (x.grad - x_ref.grad).abs().max().item())

    # 权重梯度：各 rank 的分片拼起来应等于参考版
    g1 = [torch.zeros_like(col.weight.grad) for _ in range(world_size)]
    dist.all_gather(g1, col.weight.grad)
    g2 = [torch.zeros_like(row.weight.grad) for _ in range(world_size)]
    dist.all_gather(g2, row.weight.grad)
    if rank == 0:
        full_g1 = torch.cat(g1, dim=0)
        full_g2 = torch.cat(g2, dim=1)
        print('第一层权重梯度（拼接）最大差:', (full_g1 - l1.weight.grad).abs().max().item())
        print('第二层权重梯度（拼接）最大差:', (full_g2 - l2.weight.grad).abs().max().item())
        print('（四项全 ≈0：TP 与单卡严格等价，f/g 方向正确）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run, args=(2,), nprocs=2, join=True)


## 3. 为什么这样组合只通信一次

对着代码数通信点：`col` 的出口没有通信（输出保持分片）、GELU 逐元素不跨维、`row` 的出口 `g_op` 做唯一一次 all-reduce。反向同理：梯度回流时 `g` 恒等、`f` 做唯一一次 all-reduce。一个 MLP block 前向 1 次、反向 1 次 —— 对比"每层都 all-gather 拼全量"的朴素切法（每层 2 次+），这就是 Megatron 省钱的地方。

Attention 同理：QKV 投影列切（按 head 分片天然契合），输出投影行切，每 block 出口通信一次。

## 4. 调试清单

- **症状：loss 能降但不对** —— row 出口忘 all-reduce，每 rank 只有部分和，输出量级约小 $ N $ 倍。
  **修复**：查输出量级是第一反应；row-parallel 的出口必须补上 all-reduce。
- **症状：输出整体偏掉一个常数** —— bias 加了两遍。
  **修复**：bias 只能在 row-parallel 出口加一次；column 侧每 rank 的半截 bias 是给半截输出的，不能求和后再加。
- **症状：前向数值全对、梯度全错** —— f/g 写反。
  **修复**：前向对拍放行、梯度对拍现形 —— 验证必须带梯度（第 2 节的对拍三件套就是为这个准备的）。
- **症状：拼起来根本不是同一个 $ W $ ** —— 分片初始化种子不一致。
  **修复**：每 rank 用同一种子初始化全量权重再取自己那片，或 rank 0 广播。

## 小结

- TP 的全部手感就两句话：列切不通信、行切出口求和；MLP 用"列切→GELU→行切"把通信压到每 block 一次。
- f/g 是骨架：前向/反向一个恒等一个 all-reduce，方向写反只有梯度对拍能抓。
- TP 吃节点内 NVLink（频繁、小消息、延迟敏感），DP 吃节点间网络（每步一次、大消息）—— 通信拓扑要和硬件层次对齐，这是并行策略组合的总原则。

**下一篇**：RL Systems 07 — rollout→train 的系统骨架。